<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sl-edu-reform-voices/blob/main/Solution_2410215.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilingual NLP Analysis of Public YouTube Discussion on Sri Lankan School Education Reforms
**Student ID:** 2410215 | **Module:** CM4608 Natural Language Processing

This notebook collects public YouTube comments about the Sri Lankan school education reforms (English, Sinhala and Tamil) and analyses them with NLP: data collection and EDA (Task 1), preprocessing and tokenisation (Task 2), theme discovery (Task 3), classification (Task 4), structured extraction (Task 5), and evidence-grounded summaries with an ethics discussion (Task 6).

### 0.1 Reproducibility
A fixed random seed is used for every random step, and package versions are printed so the results can be reproduced.

## 1. Task 1 — Video discovery, comment collection and initial exploration

### 1.1 Environment set-up and API connection test

**Purpose:** Connect to the YouTube Data API v3 and confirm that the API key works before collecting any data.

**Approach:**
- Use google-api-python-client, Google’s official Python client for its APIs.
- Read saved API Key Secrets from colab
- The key is restricted in Google Cloud Console to the YouTube Data API v3 only. No OAuth or user login is used, because only public comments are collected
- Run a single test search.

1. Install the library

In [1]:
!pip install -q google-api-python-client

from google.colab import userdata
from googleapiclient.discovery import build
import sys, random, importlib.metadata as md
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
for pkg in ["numpy", "pandas", "google-api-python-client"]:
    print(f"{pkg}: {md.version(pkg)}")

Python: 3.13.18
numpy: 2.4.6
pandas: 2.2.3
google-api-python-client: 2.201.0


2. Read API Key

In [2]:
API_KEY = userdata.get("YOUTUBE_API_KEY")

3. Create the YouTube Client

In [3]:
yt = build("youtube", "v3", developerKey=API_KEY)

In [4]:
# Connection test: one search
r = yt.search().list(q="Sri Lanka education reform", part="snippet",
                     type="video", maxResults=3, regionCode="LK").execute()
for item in r["items"]:
    print(item["id"]["videoId"], "|", item["snippet"]["channelTitle"], "|", item["snippet"]["title"])

aUKv__hANms | Manoj Dhananjaya | New Education Reforms 2026 Sri Lanka | 2026 Education Reforms | GPA System for Schools | OL 2026
k7UQIyhMW2w | Deepal Warnakulasuriya | Education Reforms to be implemented from January
34UGmRqtBgE | idealisland. | Sri Lanka - Education Reforms Whitepaper? Prof. Arjuna Parakrama


4.Storage


In [5]:
from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/NLP_sl-edu-reform-voices"
RAW_DIR  = f"{PROJECT_DIR}/data/raw"        # one file per video (raw API data)
PROC_DIR = f"{PROJECT_DIR}/data/processed"  # combined datasets
DOCS_DIR = f"{PROJECT_DIR}/docs"            # search log + video lists (reproducibility)

for d in (RAW_DIR, PROC_DIR, DOCS_DIR):
    os.makedirs(d, exist_ok=True)           # create folders if missing

print("Saving data under:", PROJECT_DIR)

Mounted at /content/drive
Saving data under: /content/drive/MyDrive/NLP_sl-edu-reform-voices


### 1.2 Collection configuration and logging

**Purpose:** Record the metadata required by Task 1(a) for reproducibility: every search query, the exact date and time it was run, the number of results, and the API quota used. All timestamps are in UTC so they are unambiguous.